# Crop Production and Yield Analysis Using Python
**Student:** Krishna Singh  
**Admission No.:** 2026B01010717  
**College:** ABES Engineering College, Ghaziabad  
**Training:** HCLTech Data Analytics

**Goal:** Clean the crop dataset and compare crop production and yield using basic Pandas. **No graphs.**

**Before running in Google Colab:** Upload both `crop_production_1000.csv` and `synthetic_record_ids.csv` using the Files panel on the left. Then click **Runtime → Run all**.

*Data note:* 710 supplied rows + 290 generated (synthetic) rows. Synthetic records are examples, not additional real farm measurements.

In [ ]:
# 1. Read the dataset
import pandas as pd

df = pd.read_csv("crop_production_1000.csv")
synthetic_ids = pd.read_csv("synthetic_record_ids.csv")
print("Rows and columns:", df.shape)
print(df.head())

In [ ]:
# 2. Check for data problems
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["Farm_Record_ID"].duplicated().sum())
print("Invalid area:", (df["Area_Hectares"] <= 0).sum())
print(df.describe().round(2))

In [ ]:
# 3. Clean the data and calculate yield
clean = df.drop_duplicates().copy()
clean["Rainfall_mm"] = clean["Rainfall_mm"].fillna(clean["Rainfall_mm"].median())
clean["Soil_pH"] = clean["Soil_pH"].fillna(clean["Soil_pH"].median())
clean = clean[clean["Area_Hectares"] > 0].copy()
clean["Yield_Tons_per_Hectare"] = clean["Production_Tons"] / clean["Area_Hectares"]
clean["Is_Synthetic"] = clean["Farm_Record_ID"].isin(synthetic_ids["Farm_Record_ID"])
print("Cleaned rows:", len(clean))
print(clean["Is_Synthetic"].value_counts())

In [ ]:
# 4. Compare production and yield by crop, region, season, soil and irrigation
original = clean[clean["Is_Synthetic"] == False]
for group in ["Crop_Type", "Region", "Season", "Soil_Type", "Irrigation"]:
    table = original.groupby(group)[["Production_Tons", "Area_Hectares"]].sum()
    table["Yield_per_Hectare"] = table["Production_Tons"] / table["Area_Hectares"]
    print("\n", group, "\n", table.round(2))

In [ ]:
# 5. Check rainfall, fertilizer, soil pH and yield
print(original.groupby("Crop_Type")[["Rainfall_mm", "Fertilizer_kg_per_ha", "Soil_pH"]].mean().round(2))
print("\nYield summary:\n", original["Yield_Tons_per_Hectare"].describe().round(2))
print("\nTop 5 yields:\n", original.nlargest(5, "Yield_Tons_per_Hectare")[["Crop_Type", "Region", "Yield_Tons_per_Hectare"]])

In [ ]:
# 6. Compare original and synthetic data, then save the results
print(clean.groupby("Is_Synthetic")[["Production_Tons", "Area_Hectares"]].sum().round(2))
clean.to_csv("crop_cleaned_project.csv", index=False)
original.to_csv("crop_original_only.csv", index=False)
print("Saved: crop_cleaned_project.csv and crop_original_only.csv")

### Conclusion
We checked missing values and duplicates, cleaned the dataset, calculated yield per hectare, and compared crop production by crop type, region, season, soil and irrigation. Rainfall, fertilizer and soil pH were also summarized.

**Important:** The main comparisons use the original records only. Synthetic records are kept separately for learning. This simple version intentionally has **no graphs**, so it does not fulfill the visualization portion of the earlier SRS. These comparisons do not prove that rainfall, fertilizer or irrigation caused differences in yield.